# Exercise 6: Basic Image Generation

**CAS Agentic AI & Deep Learning - Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Implement and train a **Generative Adversarial Network (GAN)**
- Implement the **reparameterization trick** and train a **Variational Autoencoder (VAE)**
- Train a **Diffusion Model (DM)** using `diffusers` components
- Visualize and interpret samples generated by each model type

**Fill in code where you see `# YOUR CODE HERE`.**

<div style="background-color:#fff3cd;border-left:6px solid #ff9900;padding:14px 18px;margin:12px 0;border-radius:4px;color:#222">
<div style="font-size:18px;font-weight:bold">⚡ This exercise needs a GPU runtime</div>
<div style="margin-top:8px">
<b>In Google Colab:</b> click <code>Runtime → Change runtime type → T4 GPU</code> (free tier).<br>
Training generative models on CPU works but takes 10–20 minutes per section.
</div>
</div>

## Setup Colab / Local Paths

The following setup sets the default data path.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")

if IN_COLAB:
    print("Installing additional packages to Colab Environment")
    %pip install -q diffusers accelerate lightning torchinfo

    ROOT_PATH = Path("/content")
    DATA_PATH = ROOT_PATH.joinpath("data")

else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

If you want to mount your Google Drive you can use the following function. This will enable you to persist data and model checkpoints across sessions.

In [ ]:
def mount_drive(
    drive_mount_path: str = "/content/drive", data_path: str = "MyDrive/cas-dl-compvis"
) -> tuple[Path, Path]:
    from google.colab import drive

    root_path = Path(drive_mount_path)
    drive.mount(str(root_path))
    data_path = root_path.joinpath(data_path)
    return root_path, data_path


# Uncomment to mount Google Drive (persists data between sessions):
# ROOT_PATH, DATA_PATH = mount_drive()

## Helper Functions

In [ ]:
import math
import time
from textwrap import wrap

import numpy as np
import torch
from matplotlib import pyplot as plt
from torchvision.transforms.v2 import functional as TF
from torchvision.utils import make_grid


def plot_collage(
    images,
    captions=None,
    nrows=None,
    ncols=None,
    caption_width=30,
    caption_font_size=10,
    figsize=None,
    title=None,
):
    """Plot a list of PIL images or [0, 1] tensors in a grid, optionally with captions."""
    num_images = len(images)
    if nrows is None or ncols is None:
        nrows = ncols = math.ceil(math.sqrt(num_images))
    if figsize is None:
        figsize = (ncols * 2, nrows * 2)
    fig, axes = plt.subplots(figsize=figsize, nrows=nrows, ncols=ncols, squeeze=False)
    if title:
        fig.suptitle(title, fontsize=20)
    for i, ax in enumerate(axes.flatten()):
        ax.set_xticks([])
        ax.set_yticks([])
        if i >= num_images:
            ax.axis("off")
            continue
        img = images[i]
        if isinstance(img, torch.Tensor):
            img = TF.to_pil_image(img)
        ax.imshow(np.array(img), cmap="gray", vmin=0, vmax=255)
        if captions is not None and i < len(captions):
            caption = "\n".join(wrap(captions[i], caption_width))
            ax.set_title(caption, fontsize=caption_font_size)
    return fig, axes


def show_samples(samples, nrow=4, title=None):
    """Show a batch of [0, 1] image tensors (N, C, H, W) as a single grid."""
    grid = make_grid(samples.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    ncols = min(nrow, len(samples))
    nrows = math.ceil(len(samples) / nrow)
    fig, ax = plt.subplots(figsize=(ncols * 1.0, nrows * 1.0))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis("off")
    if title:
        ax.set_title(title)
    return fig, ax


def denorm(x):
    """Map images from [-1, 1] back to [0, 1] for display."""
    return (x * 0.5 + 0.5).clamp(0, 1)

## Imports

In [ ]:
from collections.abc import Callable

import lightning as L
from diffusers import DDPMPipeline, DDPMScheduler, UNet2DModel
from lightning.pytorch.loggers import TensorBoardLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchinfo import summary
from torchvision import transforms
from torchvision.datasets import MNIST

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device} | Colab: {IN_COLAB}")

if IN_COLAB and device.type == "cpu":
    from IPython.display import HTML, display

    display(
        HTML("""
    <div style="background:#ffebee;border:3px solid #d32f2f;border-radius:8px;
                padding:16px;margin:12px 0;font-size:16px;color:#000">
        <div style="font-size:20px;font-weight:bold;color:#d32f2f">
            ⚠️ NO GPU DETECTED — training will be very slow
        </div>
        <div style="margin-top:8px">
            Switch to a GPU runtime: <b>Runtime → Change runtime type → T4 GPU</b>
            (free tier), then <b>Runtime → Run all</b>.
        </div>
    </div>
    """)
    )

## Configuration and Shared Utilities

All settings that drive the runtime live in the next cell — adjust epochs there if training is too slow.

Each model section below (GAN, VAE, DDPM) only depends on the cells up to and including **0) Prepare Dataset**, so you can skip sections. Every trained model is registered in `TRAINED_MODELS` (with its training time in `TIMINGS`).

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2
SEED = 123
LOG_PATH = DATA_PATH.joinpath("lightning_logs")

GAN_LATENT_DIM = 20
GAN_EPOCHS = 10

VAE_LATENT_DIM = 2
VAE_EPOCHS = 10

DDPM_EPOCHS = 5  # increase to 15+ for better quality
DDPM_MONITOR_EVERY_N_EPOCHS = 5  # DDPM sampling (1000 steps) is slow

TRAINED_MODELS = {}
TIMINGS = {}

In [ ]:
TO_TENSOR = transforms.ToTensor()  # [0, 1]
TO_PM1 = transforms.Compose([TO_TENSOR, transforms.Normalize(0.5, 0.5)])  # [-1, 1]
TO_PM1_32 = transforms.Compose([TO_PM1, transforms.Resize(32)])  # [-1, 1], 32x32


class SampleGridMonitor(L.Callback):
    """Log the images returned by `sample_fn(pl_module)` (in [0, 1]) to TensorBoard."""

    def __init__(self, sample_fn, tag="samples", nrow=4, every_n_epochs=1):
        super().__init__()
        self.sample_fn = sample_fn
        self.tag = tag
        self.nrow = nrow
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        if (trainer.current_epoch + 1) % self.every_n_epochs != 0:
            return
        pl_module.eval()
        with torch.no_grad():
            images = self.sample_fn(pl_module)
        pl_module.train()
        grid = make_grid(images, nrow=self.nrow)
        pl_module.logger.experiment.add_image(self.tag, grid, trainer.current_epoch)


def fit_model(name, model, datamodule, max_epochs, callbacks=()):
    """Train `model`, then register it and its training time under `name`."""
    trainer = L.Trainer(
        accelerator="auto",
        devices=1,
        max_epochs=max_epochs,
        logger=TensorBoardLogger(LOG_PATH, name=name.lower()),
        default_root_dir=LOG_PATH,
        callbacks=list(callbacks),
    )
    start = time.perf_counter()
    trainer.fit(model, datamodule)
    train_s = time.perf_counter() - start
    TRAINED_MODELS[name] = model
    TIMINGS[name] = {"train_s": train_s, "epochs": max_epochs}
    print(f"{name}: trained {max_epochs} epochs in {train_s:.0f}s")
    return trainer


def make_unet():
    """Small UNet for 32x32 grayscale images."""
    return UNet2DModel(
        sample_size=32,
        in_channels=1,
        out_channels=1,
        layers_per_block=2,
        block_out_channels=(32, 32, 64, 128),
        down_block_types=(
            "DownBlock2D",
            "DownBlock2D",
            "AttnDownBlock2D",
            "DownBlock2D",
        ),
        up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    )

## 0) Prepare Dataset

We use the **MNIST** dataset throughout this exercise. Its small image size (28×28 grayscale) and clean structure make it ideal for comparing generative model architectures within a reasonable training time.

In [ ]:
ds_mnist_train = MNIST(root=DATA_PATH.joinpath("mnist"), train=True, download=True)
print(f"Training samples: {len(ds_mnist_train)}")

Let's inspect a few samples from the dataset.

In [ ]:
images = [ds_mnist_train[i][0] for i in range(16)]
labels = [f"Label: {ds_mnist_train[i][1]}" for i in range(16)]
fig, ax = plot_collage(images, captions=labels)

We create a Lightning `DataModule` to handle data loading consistently across all three models.

In [ ]:
class MNISTDataModule(L.LightningDataModule):
    def __init__(
        self,
        data_dir: str,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
        transform_fn: Callable | None = None,
    ):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.transform = transform_fn or TO_TENSOR

    def prepare_data(self):
        MNIST(self.data_dir, train=True, download=True)

    def setup(self, stage=None):
        self.ds_train = MNIST(self.data_dir, train=True, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(
            self.ds_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )

In [ ]:
data_module = MNISTDataModule(data_dir=DATA_PATH.joinpath("mnist"))
data_module.setup("fit")
image, label = data_module.ds_train[0]
print(f"Image shape: {image.shape}, dtype: {image.dtype}, label: {label}")

## 1) Generative Adversarial Networks (GAN)

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

A GAN consists of two networks trained in opposition:
- **Generator** $\mathcal{G}$: maps a random latent vector $z \sim \mathcal{N}(0, I)$ to a synthetic image.
- **Discriminator** $\mathcal{D}$: predicts whether an image is real or generated.

Training alternates between:
1. **Generator step**: $\mathcal{G}$ tries to fool $\mathcal{D}$ — minimise $\mathcal{D}$'s confidence on fake images.
2. **Discriminator step**: $\mathcal{D}$ tries to distinguish real from fake — minimise classification error on both.

The minimax objective is:
$$\min_G \max_D \; \mathbb{E}_{x}[\log D(x)] + \mathbb{E}_z[\log(1 - D(G(z)))]$$

### Generator

Maps a latent vector to a 28×28 image via transposed convolutions.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=GAN_LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.model = nn.Sequential(
            nn.Linear(self.latent_dim, 6 * 49),
            nn.Unflatten(1, (6, 7, 7)),
            nn.ConvTranspose2d(
                6, 32, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.ConvTranspose2d(
                32, 64, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.Conv2d(64, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 1, kernel_size=1, stride=1, padding=0),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        return self.model(z)


generator = Generator(latent_dim=GAN_LATENT_DIM)
summary(generator, input_size=(32, GAN_LATENT_DIM))

### Discriminator

Classifies a 28×28 image as real (1) or fake (0) via convolutions.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 1),
        )

    def forward(self, img):
        return self.model(img)


discriminator = Discriminator()
summary(discriminator, input_size=(32, 1, 28, 28))

### GAN Lightning Module

**Task**: Implement `GAN.training_step`. The GAN uses manual optimization (`automatic_optimization = False`), so you explicitly call `optimizer.step()` for each sub-network.

Your training step should:
1. Sample a batch of latent vectors `z` from a standard normal distribution.
2. **Generator step**: generate fake images → compute generator loss (fool the discriminator) → backpropagate → step the generator optimizer.
3. **Discriminator step**: compute loss on real images (label ≈ 1) and fake images (label ≈ 0) → combine → backpropagate → step the discriminator optimizer.

*Hint*: Use `F.binary_cross_entropy_with_logits`. Use `self.toggle_optimizer` / `self.untoggle_optimizer` around each sub-step as shown in the Lightning docs.

In [ ]:
class GAN(L.LightningModule):
    def __init__(self, latent_dim=GAN_LATENT_DIM):
        super().__init__()
        self.save_hyperparameters()
        self.generator = Generator(latent_dim=latent_dim)
        self.discriminator = Discriminator()
        self.latent_dim = latent_dim
        self.automatic_optimization = False  # we manage optimizers manually

    def forward(self, z):
        return self.generator(z)

    def training_step(self, batch, batch_idx):
        x_real, _ = batch
        optimizer_g, optimizer_d = self.optimizers()

        # Sample latent vectors
        z = torch.randn((x_real.shape[0], self.latent_dim), device=self.device)

        # ==== Train the Generator ====
        self.toggle_optimizer(optimizer_g)
        x_fake = self.generator(z)
        logits_fake = self.discriminator(x_fake)
        y_true = torch.ones_like(logits_fake)
        g_loss = F.binary_cross_entropy_with_logits(logits_fake, y_true)
        self.log("g_loss", g_loss, prog_bar=True)
        self.manual_backward(g_loss)
        optimizer_g.step()
        optimizer_g.zero_grad()
        self.untoggle_optimizer(optimizer_g)

        # ==== Train the Discriminator ====
        self.toggle_optimizer(optimizer_d)
        # Real images: label ≈ 1 (with small noise for stability)
        logits_real = self.discriminator(x_real)
        y_real = 1.0 - torch.rand_like(logits_real) * 0.05
        d_real_loss = F.binary_cross_entropy_with_logits(logits_real, y_real)
        # Fake images: label ≈ 0
        logits_fake = self.discriminator(x_fake.detach())
        y_fake = torch.rand_like(logits_fake) * 0.05
        d_fake_loss = F.binary_cross_entropy_with_logits(logits_fake, y_fake)
        d_loss = 0.5 * (d_real_loss + d_fake_loss)
        self.log("d_loss", d_loss, prog_bar=True)
        self.manual_backward(d_loss)
        optimizer_d.step()
        optimizer_d.zero_grad()
        self.untoggle_optimizer(optimizer_d)

    def configure_optimizers(self):
        return (
            torch.optim.Adam(self.generator.parameters(), lr=1e-3),
            torch.optim.Adam(self.discriminator.parameters(), lr=1e-3),
        )

**Question**: What loss value do you expect for a well-trained GAN where both networks are equally strong?

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir={LOG_PATH} --host 0.0.0.0 --port=6006

In [ ]:
L.seed_everything(SEED)

# Normalize to [-1, 1] to match the Tanh output of the generator
data_module_gan = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_PM1
)
gan_model = GAN(latent_dim=GAN_LATENT_DIM)


def gan_monitor_samples(model, n=16):
    z = torch.randn(n, model.latent_dim, device=model.device)
    return denorm(model(z))


fit_model(
    "GAN",
    gan_model,
    data_module_gan,
    max_epochs=GAN_EPOCHS,
    callbacks=[SampleGridMonitor(gan_monitor_samples, tag="gan/samples")],
)

**Task**: Sample 16 images from the trained GAN and display them using `plot_collage`.

In [ ]:
gan_model.eval()
z = torch.randn(16, GAN_LATENT_DIM, device=gan_model.device)
with torch.no_grad():
    samples = gan_model(z)

# Denormalize from [-1, 1] → [0, 1] for display
fig, ax = show_samples(denorm(samples), nrow=4, title="GAN samples")

## 2) Variational Autoencoders (VAE)

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

A VAE learns a probabilistic latent space by optimising the **Evidence Lower Bound (ELBO)**:

$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{D_{\text{KL}}(q(z|x) \| p(z))}_{\text{regularisation}}$$

The encoder maps $x$ to a distribution $q(z|x) = \mathcal{N}(\mu, \sigma^2)$; the decoder maps samples $z$ back to the image space.

The key trick enabling backpropagation through the sampling step is the **reparameterisation trick**:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

### Task 1 — Reparameterisation Trick

Implement `reparameterize(z_mean, z_log_var)`. Given the mean and log-variance of the latent distribution, return a sample using the reparameterisation trick.

*Hint*: `std = exp(0.5 * z_log_var)`, then sample `eps ~ N(0,I)` using `torch.randn_like(std)`.

In [ ]:
def reparameterize(z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    """Sample from q(z|x) = N(z_mean, exp(z_log_var)) using the reparameterisation trick."""
    std = torch.exp(0.5 * z_log_var)
    eps = torch.randn_like(std)
    return z_mean + eps * std

In [ ]:
# Quick sanity check: the output should have the same shape as z_mean
z_mean_test = torch.zeros(4, 8)
z_log_var_test = torch.zeros(4, 8)
z_sample = reparameterize(z_mean_test, z_log_var_test)
assert z_sample.shape == z_mean_test.shape, (
    f"Expected {z_mean_test.shape}, got {z_sample.shape}"
)
print("OK")

### Encoder and Decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(7 * 7 * 64, 16),
            nn.ReLU(),
        )
        self.z_mean = nn.Linear(16, latent_dim)
        self.z_log_var = nn.Linear(16, latent_dim)

    def forward(self, x):
        pre_z = self.network(x)
        z_mean = self.z_mean(pre_z)
        z_log_var = self.z_log_var(pre_z)
        z = reparameterize(z_mean, z_log_var)
        return z_mean, z_log_var, z


class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(latent_dim, 7 * 7 * 64), nn.ReLU())
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(
                64, 64, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.ReLU(),
            nn.ConvTranspose2d(
                64, 32, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),  # output in [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 64, 7, 7)
        return self.deconv(x)


encoder = Encoder(latent_dim=VAE_LATENT_DIM)
summary(encoder, input_size=(16, 1, 28, 28))

### Task 2 — VAE Training Step

Implement `VAE.training_step`. Given a batch `(x, _)`:

1. Pass `x` through the encoder to get `z_mean`, `z_log_var`, `z`.
2. Decode `z` to get the reconstruction `x_rec`.
3. Compute the **reconstruction loss** (binary cross-entropy per pixel, summed over pixel dimensions, averaged over the batch).
4. Compute the **KL divergence** loss:
   $\mathcal{L}_{KL} = -\frac{1}{2} \sum_j (1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2)$,
   then divide by batch size.
5. Return the sum `rec_loss + kl_loss`. Log all three scalars.

In [ ]:
class VAE(L.LightningModule):
    def __init__(self, latent_dim=VAE_LATENT_DIM):
        super().__init__()
        self.save_hyperparameters()
        self.encoder = Encoder(latent_dim=latent_dim)
        self.decoder = Decoder(latent_dim=latent_dim)

    def forward(self, z):
        return self.decoder(z)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        z_mean, z_log_var, z = self.encoder(x)
        x_rec = self.decoder(z)
        rec_loss = (
            F.binary_cross_entropy(x_rec, x, reduction="none").sum(dim=(1, 2, 3)).mean()
        )
        kl_loss = -0.5 * torch.sum(1 + z_log_var - z_mean**2 - torch.exp(z_log_var))
        kl_loss = kl_loss / x.shape[0]
        loss = rec_loss + kl_loss
        self.log("rec_loss", rec_loss, prog_bar=True)
        self.log("kl_loss", kl_loss, prog_bar=True)
        self.log("loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [ ]:
L.seed_everything(SEED)

# The decoder ends with a Sigmoid, so the VAE works on images in [0, 1]
data_module_vae = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_TENSOR
)
data_module_vae.setup("fit")
monitor_images = torch.stack([data_module_vae.ds_train[i][0] for i in range(8)])

vae_model = VAE(latent_dim=VAE_LATENT_DIM)


def vae_monitor_samples(model, n=16):
    z = torch.randn(n, VAE_LATENT_DIM, device=model.device)
    return model(z)


def vae_monitor_reconstructions(model):
    x = monitor_images.to(model.device)
    _, _, z = model.encoder(x)
    return torch.cat([x, model.decoder(z)], dim=0)


fit_model(
    "VAE",
    vae_model,
    data_module_vae,
    max_epochs=VAE_EPOCHS,
    callbacks=[
        SampleGridMonitor(vae_monitor_samples, tag="vae/samples"),
        SampleGridMonitor(
            vae_monitor_reconstructions,
            tag="vae/reconstructions",
            nrow=len(monitor_images),
        ),
    ],
)

### Latent Space Exploration

With a 2-D latent space we can visualise the entire learned manifold by decoding points on a regular grid.

In [ ]:
@torch.no_grad()
def get_latent_space_samples(
    decoder, num_points_per_dim=20, latent_range=3.0, device="cpu"
):
    grid = torch.linspace(-latent_range, latent_range, num_points_per_dim)
    xs, ys = torch.meshgrid(grid, grid, indexing="xy")
    z = torch.stack([xs.flatten(), ys.flatten()], dim=1).to(device)
    return decoder(z).cpu()


vae_model.eval()
samples = get_latent_space_samples(
    vae_model.decoder, num_points_per_dim=20, device=vae_model.device
)
fig, ax = show_samples(samples, nrow=20, title="VAE latent space (2-D grid)")

**Question**: Without using any labels during training, the latent space organises digit identity spatially. Why does this happen?

## 3) Diffusion Models (DDPM)

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

Diffusion Models (DMs) learn to reverse a gradual noising process:

**Forward process** (fixed): $q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) I)$

**Backward process** (learned): A UNet $\epsilon_\theta(x_t, t)$ predicts the noise added at step $t$.

**Training objective**: minimise the mean-squared error between the true noise $\epsilon$ and the predicted noise $\hat{\epsilon}_\theta(x_t, t)$:
$$\mathcal{L}_{\text{DM}} = \mathbb{E}_{x_0, \epsilon, t} \left[ \| \epsilon - \hat{\epsilon}_\theta(x_t, t) \|^2 \right]$$

### Visualising the Forward Process

Let's see how the scheduler progressively destroys an image.

In [ ]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# Load a single image (scaled to [-1, 1], 32x32) and add noise at increasing timesteps
x0 = TO_PM1_32(ds_mnist_train[0][0]).unsqueeze(0)

steps = [0, 100, 250, 500, 750, 999]
noised_images = []
for t in steps:
    eps = torch.randn_like(x0)
    x_t = noise_scheduler.add_noise(x0, eps, torch.tensor([t]))
    noised_images.append(denorm(x_t.squeeze(0)))

fig, ax = plot_collage(
    noised_images, captions=[f"t={t}" for t in steps], nrows=1, ncols=6
)

### UNet Denoiser

We use a small `UNet2DModel` from `diffusers` as our denoising network $\epsilon_\theta$. It is built by `make_unet()` in the shared utilities.

In [ ]:
print(f"UNet parameters: {sum(p.numel() for p in make_unet().parameters()):,}")

### Task 3 — DM Training Step

Implement `DM.training_step`. For each batch:

1. Sample random **timesteps** $t$ uniformly from $[0, T)$ — one per image.
2. Sample **Gaussian noise** $\epsilon$ with the same shape as `x_zero`.
3. **Add noise** to `x_zero` using `self.noise_scheduler.add_noise(x_zero, eps, timesteps)` to get $x_t$.
4. **Predict the noise** using `self.denoising(x_t, timesteps)["sample"]`.
5. Compute the **MSE loss** between the predicted and true noise.
6. Log and return the loss.

In [ ]:
class DM(L.LightningModule):
    def __init__(self, T=1000, beta_0=1e-4, beta_T=0.02):
        super().__init__()
        self.save_hyperparameters()
        self.noise_scheduler = DDPMScheduler(
            num_train_timesteps=T, beta_start=beta_0, beta_end=beta_T
        )
        self.denoising = make_unet()

    def training_step(self, batch, batch_idx):
        x_zero, _ = batch
        batch_size = x_zero.shape[0]

        # Sample random timesteps
        timesteps = torch.randint(
            0,
            self.noise_scheduler.config.num_train_timesteps,
            (batch_size,),
            device=x_zero.device,
        )

        # Sample noise
        eps = torch.randn_like(x_zero)

        # Add noise to get x_t
        x_t = self.noise_scheduler.add_noise(x_zero, eps, timesteps)

        # Predict noise
        noise_estimate = self.denoising(x_t, timesteps)["sample"]

        # MSE between predicted and actual noise
        loss = F.mse_loss(noise_estimate, eps)
        self.log("loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [ ]:
@torch.no_grad()
def sample_ddpm(model, n=16):
    """Run the full DDPM reverse process and return n images in [0, 1]."""
    pipe = DDPMPipeline(unet=model.denoising, scheduler=model.noise_scheduler)
    pipe.set_progress_bar_config(disable=True)
    images = pipe(batch_size=n, output_type="np").images  # (N, H, W, C) in [0, 1]
    return torch.from_numpy(images).permute(0, 3, 1, 2)

In [ ]:
L.seed_everything(SEED)

data_module_dm = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_PM1_32
)
dm_model = DM()

fit_model(
    "DDPM",
    dm_model,
    data_module_dm,
    max_epochs=DDPM_EPOCHS,
    callbacks=[
        SampleGridMonitor(
            sample_ddpm,
            tag="ddpm/samples",
            every_n_epochs=DDPM_MONITOR_EVERY_N_EPOCHS,
        )
    ],
)

### Sample from the Trained Diffusion Model

In [ ]:
dm_model.to(device).eval()  # Lightning moves the model back to CPU after fit
fig, ax = show_samples(sample_ddpm(dm_model), nrow=4, title="DDPM samples")

**Questions**:
1. How does the quality of GAN, VAE, and DM samples compare after the same number of training epochs?
2. What happens to GAN training if the discriminator becomes too strong early on?
3. How does the DDPM sampling speed compare to GAN/VAE, and what is the trade-off?